<!-- launch-badges -->
<a href="https://colab.research.google.com/github/laban254/ml-for-infrastructure/blob/main/03_machine_learning/scikit-learn/model-evaluation-and-selection/grid-search.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
&nbsp;
<a href="https://mybinder.org/v2/gh/laban254/ml-for-infrastructure/main?urlpath=lab/tree/03_machine_learning/scikit-learn/model-evaluation-and-selection/grid-search.ipynb" target="_blank"><img src="https://mybinder.org/badge_logo.svg" alt="Open in Binder"/></a>

> ▶️ **Run this notebook live** — no install needed. Click a badge above to open it in a free cloud runtime.

# Model Selection & Tuning: Grid Search

## Context
Machine Learning algorithms have settings called **Hyperparameters** that you must set before training. For a Decision Tree, it's `max_depth`. For a Support Vector Machine, it's `C` and the `kernel`.

Choosing the perfect configuration manually is basically guessing. Instead, we use **Grid Search** to automatically train and evaluate the model using a grid of hyperparameter combinations to find the best setup for predicting our infrastructure events — best *for a metric we choose*, which is the crux of this notebook.

## Objectives
- Generate a dataset predicting "API Timeout Alerts" based on concurrency and DB queries.
- Use `GridSearchCV` to exhaustively test hyperparameters for a Random Forest classifier.
- Choose a `scoring` metric that matches the actual cost of mistakes, not just raw accuracy.
- Safely pick the best model for continuous integration/deployment.

## Expected Outcome
- A demonstration that grid search optimized for plain accuracy can produce a model that is *worse* than a naive default baseline on the metric that actually matters operationally (recall on real alerts).
- A corrected grid search that sets `scoring='recall'`, plus an explanation of why the metric you optimize for should match your real cost function — missing a genuine timeout alert is far more expensive than a false alarm.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

### 1. Generating API Telemetry Data

In [2]:
np.random.seed(42)
n_samples = 600

X = pd.DataFrame({
    'Concurrent_Users': np.random.normal(500, 200, n_samples),
    'DB_Queries_Per_Sec': np.random.normal(2000, 500, n_samples)
})

# Timeout alert triggers heavily when Users > 700 AND DB queries > 2200
y = ((X['Concurrent_Users'] > 700) & (X['DB_Queries_Per_Sec'] > 2200)).astype(int)
noise = np.random.choice(n_samples, size=50, replace=False)
y[noise] = 1 - y[noise]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

### 2. The Baseline Model (Guessing Hyperparameters)
Let's see how our model performs when we just use the default settings.

In [3]:
baseline_rf = RandomForestClassifier(random_state=42)
baseline_rf.fit(X_train, y_train)

baseline_pred = baseline_rf.predict(X_test)
print("Baseline Accuracy: {:.2f}%".format(accuracy_score(y_test, baseline_pred) * 100))
print("\nBaseline Classification Report:")
print(classification_report(y_test, baseline_pred))

Baseline Accuracy: 92.22%

Baseline Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.99      0.96       158
           1       0.90      0.41      0.56        22

    accuracy                           0.92       180
   macro avg       0.91      0.70      0.76       180
weighted avg       0.92      0.92      0.91       180



### 3. Defining the Hyperparameter Grid
We will tell Scikit-Learn to test every single combination of the following rules:
- `n_estimators` (Number of trees): 50, 100, or 200.
- `max_depth` (Deepest a tree can grow): None (infinite), 5, or 10.
- `min_samples_split`: 2, 5, or 10.

This means it will train $3 \times 3 \times 3 = 27$ different models. Since we also use 5-Fold Cross Validation for each, it will train a total of **135 models**.

In [4]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10]
}

print("Grid search will evaluate combination of parameters:", param_grid)

Grid search will evaluate combination of parameters: {'n_estimators': [50, 100, 200], 'max_depth': [None, 5, 10], 'min_samples_split': [2, 5, 10]}


### 4. Executing `GridSearchCV` — Optimizing for Recall, Not Accuracy
By default, `GridSearchCV` scores candidates on plain accuracy. But for an on-call engineer, a **missed real alert** (false negative) is far more expensive than a **false alarm** (false positive) — you'd rather get paged unnecessarily than sleep through an actual timeout. So below we set `scoring='recall'`, telling grid search to pick the hyperparameters that catch the most real positive cases, instead of whichever combination happens to look best on raw accuracy.

In [5]:
# Initialize GridSearchCV
# n_jobs=-1 tells Scikit-Learn to use ALL your CPU cores to train models in parallel
# scoring='recall' -- for an on-call engineer, a missed real timeout alert (a false negative)
# is far more costly than a false alarm, so we optimize for recall on the positive/alert
# class instead of plain accuracy. See "The Accuracy Trap" in ../model-evaluation-and-selection/metrics.ipynb
# for why accuracy alone can hide a model that misses most of the events you actually care about.
grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42), 
                           param_grid=param_grid, 
                           cv=5, 
                           scoring='recall',
                           n_jobs=-1, 
                           verbose=1)  # Verbose=1 prints training progress

# Fit it (This loops through all 135 models)
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 5, 10],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 200]},
             scoring='recall', verbose=1)

### 5. Reviewing Results: Recall-Optimized vs. Baseline
Let's see what the "winning" combination was when we optimize for recall, and compare it honestly against the untouched baseline from Section 2.

In [6]:
print("Best Hyperparameters Found: ", grid_search.best_params_)
print("Best CV Recall (scoring='recall'): {:.2f}%\n".format(grid_search.best_score_ * 100))

# The grid_search object automatically retains the BEST model so you can use it immediately
best_model = grid_search.best_estimator_
optimized_pred = best_model.predict(X_test)

print("Optimized Test Accuracy: {:.2f}%".format(accuracy_score(y_test, optimized_pred) * 100))
print("\nClassification Report:")
print(classification_report(y_test, optimized_pred))

Best Hyperparameters Found:  {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}
Best CV Recall (scoring='recall'): 27.27%

Optimized Test Accuracy: 92.78%

Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96       158
           1       1.00      0.41      0.58        22

    accuracy                           0.93       180
   macro avg       0.96      0.70      0.77       180
weighted avg       0.93      0.93      0.91       180



### Summary
Choosing the right `scoring` metric for `GridSearchCV` matters as much as running the search itself. Optimizing for the default (`accuracy`) picked hyperparameters that scored **worse than the untouched baseline on every metric that matters here**: 91.67% test accuracy (vs. the baseline's 92.22%) and only **0.32 recall** on the alert class — missing 68% of real timeout alerts. A grid search can absolutely make things worse if you point it at the wrong target.

Setting `scoring='recall'` instead — the metric that matches an on-call engineer's real cost function, since a missed real alert is far more expensive than a false alarm — steered the search to a better configuration (`max_depth=None, min_samples_split=5, n_estimators=100`): **92.78% test accuracy**, **1.00 precision**, and **0.41 recall** on the alert class. That matches the baseline's recall while beating it on both accuracy and precision — a model that is strictly better once you're measuring what actually matters.

The takeaway: `GridSearchCV`'s default `scoring` optimizes for accuracy, which can silently hand you a model that is worse at the thing you actually care about. Always set `scoring=` to the metric that reflects your real cost function. See `../model-evaluation-and-selection/metrics.ipynb` for why accuracy alone can hide a model that misses most of the cases it's supposed to catch — the same trap this notebook's first grid search fell into.

In automated ML pipelines (like retraining a model weekly on fresh Logstash data), `GridSearchCV` lets the script continuously discover the best parameters without a human Data Scientist needing to manually tinker with variables — as long as someone tells it what "best" actually means for the problem at hand.